# 01 - Setup & Data Access

**AI-Based Autonomous System Validation & Reliability Checker**
Predictive Disk-Failure Self-Healing Agent for Server Fleets

This notebook builds the dataset foundation. It does **not** download the 12 GB Backblaze
quarterly archive. Instead it queries Backblaze's Apache Iceberg copy of Drive Stats
directly over S3 with DuckDB, and materialises only a compact, column-pruned parquet
panel onto Google Drive.

## What this notebook produces

| Artifact | Description | Approx size |
|---|---|---|
| `drive_registry.parquet` | One row per drive: model, capacity, first/last seen, ever-failed, censoring | ~15 MB |
| `cohort.parquet` | Selected drives (all failures + stratified healthy sample) | ~2 MB |
| `panel/panel_YYYY_MM.parquet` | Per-drive **daily** telemetry for the cohort, one file per month | ~300-600 MB total |
| `fleet_minimal/fleet_YYYY_MM.parquet` | Full-fleet id + topology + failure only, for the Part 6 replay | ~150-300 MB |

## Two design decisions that matter

**1. Drive-level sampling, never row-level.** A previous attempt at this dataset sampled
*rows*, which left 114,767 of 243,926 drives with a single observed day and silently
reduced every rolling-window feature to a constant zero. We sample whole **drives** and
keep their complete daily history, so the per-drive time series stays intact. A sampling
weight is recorded per drive so metrics can be reweighted back to true fleet prevalence.

**2. A December 2025 warm-up window.** Trailing 30-day features for early-January rows
are undefined unless December is present. We pull December 2025 unlabelled purely as
lookback history. Without it, January features are silently truncated.

## Verified facts (checked against the live Iceberg table)

- The Iceberg endpoint works; a single-day predicate-pushdown query returns in ~16 s.
- On 2026-03-15: **343,446** drives, 8 failures.
- Topology is **fully populated**: `datacenter`, `cluster_id`, `vault_id`, `pod_id` at 100%,
  `pod_slot_num` at 98.7%, and `is_legacy_format` is 0 for every row.
- Geometry matches Backblaze's published vault architecture: **244 vaults with exactly 20 pods**,
  and **11,848 tomes containing exactly 20 drives**.

That last point is what makes this project more than a classifier. A *tome* is the 20 drives
sharing a `pod_slot_num` across a vault's 20 pods, protected by Reed-Solomon 17+3
(also 16+4 and 15+5 on larger drives). So the safety rule from the problem statement -
"never drain the last healthy node" - becomes a precise, checkable invariant:
**unavailable drives per tome must never reach that tome's parity count.**

## 0. Install dependencies

DuckDB does the heavy lifting: it reads the Iceberg table over S3 and pushes date
predicates and column projections down so we only transfer what we ask for.

In [ ]:
# The duckdb floor matters: older builds resolve a stale Iceberg metadata snapshot and then
# return an EMPTY table with no error, which looks exactly like a broken query.
!pip install -q --upgrade "duckdb>=1.4" pyarrow polars

import duckdb, pyarrow, polars as pl
print("duckdb ", duckdb.__version__)
print("pyarrow", pyarrow.__version__)
print("polars ", pl.__version__)

assert tuple(int(x) for x in duckdb.__version__.split(".")[:2]) >= (1, 4), (
    f"duckdb {duckdb.__version__} is too old for reliable Iceberg snapshot resolution. "
    "Restart the runtime after the pip install above so the new version is picked up.")

## 1. Configuration

Everything downstream reads from this one cell, so date windows and paths are changed
here and nowhere else.

### On the credentials

Backblaze **publishes** a read-only application key for the Drive Stats dataset on its
public documentation page, precisely so that anyone can query it. They are not secrets and
they grant nothing except read access to this one public bucket. They are still read from
the environment first so you can override them without editing the notebook.

In [ ]:
import os
from datetime import date

# ---------------------------------------------------------------- Backblaze B2 / Iceberg
B2_KEY_ID   = os.environ.get("B2_KEY_ID",   "0045f0571db506a0000000017")
B2_SECRET   = os.environ.get("B2_SECRET",   "K004Fs/bgmTk5dgo6GAVm2Waj3Ka+TE")
B2_REGION   = "us-west-004"
B2_ENDPOINT = "s3.us-west-004.backblazeb2.com"
ICEBERG_URI = "s3://drivestats-iceberg/drivestats"

# ---------------------------------------------------------------- Time windows
# December 2025 is warm-up ONLY (unlabelled lookback so 30d features are valid in January).
WARMUP_START = date(2025, 12,  1)
Q1_START     = date(2026,  1,  1)
Q1_END       = date(2026,  3, 31)

# Temporal split with an embargo gap >= the longest prediction horizon, so that a training
# row's future-looking label cannot peek into the evaluation window.
TRAIN_START, TRAIN_END = date(2026, 1, 1),  date(2026, 2, 15)
EMBARGO_START, EMBARGO_END = date(2026, 2, 16), date(2026, 2, 22)   # discarded entirely
CALIB_START, CALIB_END = date(2026, 2, 23), date(2026, 3,  8)
TEST_START,  TEST_END  = date(2026, 3,  9), date(2026, 3, 31)

# ---------------------------------------------------------------- Cohort sampling
N_HEALTHY_DRIVES = 45_000   # healthy drives to sample; ALL failed drives are always kept
RANDOM_SEED      = 42

# ---------------------------------------------------------------- Paths
IN_COLAB   = os.path.exists("/content")
DRIVE_ROOT = "/content/drive/MyDrive/hpe_capstone" if IN_COLAB else os.path.expanduser("~/hpe_capstone")
PANEL_DIR  = f"{DRIVE_ROOT}/panel"
FLEET_DIR  = f"{DRIVE_ROOT}/fleet_minimal"
REGISTRY   = f"{DRIVE_ROOT}/drive_registry.parquet"
COHORT     = f"{DRIVE_ROOT}/cohort.parquet"

print(f"in colab   : {IN_COLAB}")
print(f"drive root : {DRIVE_ROOT}")
print(f"warm-up    : {WARMUP_START} .. {Q1_START}")
print(f"labelled   : {Q1_START} .. {Q1_END}")

## 2. Mount Google Drive

Artifacts go to Drive so a Colab disconnect does not destroy hours of extraction work.
The materialisation loop in section 6 is resumable for the same reason.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

for d in (DRIVE_ROOT, PANEL_DIR, FLEET_DIR):
    os.makedirs(d, exist_ok=True)

print("ready:", DRIVE_ROOT)

## 3. Connect to the Drive Stats Iceberg table

`unsafe_enable_version_guessing` is required: it lets DuckDB inspect the table's metadata
files and pick the current snapshot. Without it the scan fails, because we are pointing at
a bare table location rather than a catalog that could tell us the version.

In [ ]:
def connect_drivestats(verbose: bool = True) -> duckdb.DuckDBPyConnection:
    """Connect and expose the Drive Stats Iceberg table as `drivestats`, snapshot pinned."""
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs; INSTALL iceberg; LOAD iceberg;")
    con.execute(
        "CREATE OR REPLACE SECRET b2 (TYPE s3, KEY_ID ?, SECRET ?, REGION ?, ENDPOINT ?);",
        [B2_KEY_ID, B2_SECRET, B2_REGION, B2_ENDPOINT])
    con.execute("SET unsafe_enable_version_guessing = true;")

    # Resolve the newest snapshot explicitly instead of leaving the choice to `version => '?'`.
    # Guessing is what silently produced a zero-row table: it picked an old metadata file, the
    # scan succeeded, and every count came back 0 with no error raised anywhere.
    snap = con.execute(
        f"SELECT snapshot_id, timestamp_ms FROM iceberg_snapshots('{ICEBERG_URI}', "
        f"version => '?') ORDER BY sequence_number DESC LIMIT 1").fetchone()
    if snap is None:
        raise RuntimeError("no Iceberg snapshots returned - cannot resolve the table")
    snapshot_id, committed = snap
    if verbose:
        print(f"pinned snapshot {snapshot_id} (committed {committed})")

    con.execute(
        f"CREATE OR REPLACE VIEW drivestats AS SELECT * FROM iceberg_scan("
        f"'{ICEBERG_URI}', snapshot_from_id => {snapshot_id}, allow_moved_paths => true);")
    return con


con = connect_drivestats()

# Probe the two boundary dates, deliberately NOT a global min(date)/max(date). The table spans
# 2013 to the present, so an unfiltered aggregate has to touch every year and takes many
# minutes; a date-equality predicate is pushed down to one day's files and returns in seconds.
for label, d in (("window start", WARMUP_START), ("window end", Q1_END)):
    n = con.execute("SELECT count(*) FROM drivestats WHERE date = ?", [d]).fetchone()[0]
    print(f"probe {label} {d}: {n:,} rows")
    if n == 0:
        raise RuntimeError(
            f"{d} returned ZERO rows. This is the stale-snapshot failure mode: the scan "
            "succeeds but the resolved metadata predates the data, so every downstream count "
            "looks like a plausible zero instead of an error. Upgrade duckdb "
            "(pip install -U duckdb), restart the runtime, and re-run.")
    if n < 100_000:
        raise RuntimeError(f"only {n:,} rows on {d}; expected >300k. Investigate first.")

print("connected and validated")

## 4. Viability checks

Run these before committing to a long extraction. They confirm the connection works, the
data is present for our window, and - critically - that the topology columns the safety
simulator depends on are actually populated for 2026 rows. The published schema file is
stamped "Last Updated Q2 2024", so this cannot be assumed.

In [ ]:
import time

t0 = time.time()
check = con.sql("""
    SELECT
        count(*)                          AS rows,
        count(DISTINCT serial_number)     AS drives,
        sum(failure)                      AS failures,
        count(datacenter)                 AS has_datacenter,
        count(cluster_id)                 AS has_cluster,
        count(vault_id)                   AS has_vault,
        count(pod_id)                     AS has_pod,
        count(pod_slot_num)               AS has_slot,
        sum(CASE WHEN is_legacy_format THEN 1 ELSE 0 END) AS legacy_rows
    FROM drivestats
    WHERE date = DATE '2026-03-15'
""").df().T.rename(columns={0: "value"})
print(check)
print(f"\nelapsed {time.time() - t0:.1f}s")

row = check["value"]
assert row["rows"] > 300_000, "unexpectedly few rows - is the snapshot resolving correctly?"
assert row["has_vault"] / row["rows"] > 0.95, "vault_id sparsely populated: safety simulator needs a fallback"
print("\nviability OK - topology is populated, proceed")

### 4b. Tome geometry

This is the structural check that underpins Part 6. We expect to see a large mode at
exactly **20 drives** per `(vault_id, pod_slot_num)` group and **20 pods** per vault.

Note the groups that come back with ~40 drives: a `(vault, slot)` pair is not guaranteed
unique per pod in every vault, so the simulator must key tomes on the triple
`(vault_id, pod_id, pod_slot_num)` for identity and group by `(vault_id, pod_slot_num)`
for the redundancy set. The third query below measures how often that matters.

In [ ]:
print("=== drives per (vault_id, pod_slot_num) ===")
print(con.sql("""
    SELECT cnt AS drives_in_group, count(*) AS n_groups
    FROM (
        SELECT vault_id, pod_slot_num, count(*) AS cnt
        FROM drivestats
        WHERE date = DATE '2026-03-15' AND pod_slot_num IS NOT NULL
        GROUP BY 1, 2
    )
    GROUP BY 1 ORDER BY n_groups DESC LIMIT 10
""").df().to_string(index=False))

print("\n=== pods per vault ===")
print(con.sql("""
    SELECT pods, count(*) AS n_vaults
    FROM (
        SELECT vault_id, count(DISTINCT pod_id) AS pods
        FROM drivestats
        WHERE date = DATE '2026-03-15' AND vault_id IS NOT NULL
        GROUP BY 1
    )
    GROUP BY 1 ORDER BY n_vaults DESC LIMIT 10
""").df().to_string(index=False))

print("\n=== is (vault_id, pod_id, pod_slot_num) unique on a given day? ===")
print(con.sql("""
    SELECT dupes AS drives_per_triple, count(*) AS n_groups
    FROM (
        SELECT vault_id, pod_id, pod_slot_num, count(*) AS dupes
        FROM drivestats
        WHERE date = DATE '2026-03-15' AND pod_slot_num IS NOT NULL
        GROUP BY 1, 2, 3
    )
    GROUP BY 1 ORDER BY drives_per_triple LIMIT 10
""").df().to_string(index=False))

### 4c. Quarter totals and the HDD / SSD split

Backblaze's published Q1 2026 figures are 30,203,180 drive-days, 341,263 drives and
1,030 failures. Matching them confirms we are reading the whole quarter.

SSDs live in the **same table** as HDDs. They have different failure physics and do not
report the same SMART attributes, so they must be separated rather than silently mixed in.
Drive Stats has no `drive_type` column, so we identify them by model-name prefix and by the
absence of rotational attributes.

In [ ]:
t0 = time.time()
print("=== Q1 2026 totals (this scans the quarter - expect a few minutes) ===")
print(con.sql(f"""
    SELECT count(*) AS drive_days,
           count(DISTINCT serial_number) AS drives,
           sum(failure) AS failures,
           count(DISTINCT date) AS days
    FROM drivestats
    WHERE date BETWEEN DATE '{Q1_START}' AND DATE '{Q1_END}'
""").df().to_string(index=False))
print(f"elapsed {time.time() - t0:.1f}s")

In [ ]:
# smart_1 (read error rate) and smart_9 (power-on hours) are effectively universal on HDDs.
# Rotational-only attributes such as smart_3 (spin-up time) and smart_193 (load cycle count)
# are absent on SSDs, which is a more reliable signal than the model string alone.
print("=== candidate SSD models on a reference day ===")
print(con.sql("""
    SELECT model,
           count(*) AS drives,
           round(100.0 * count(smart_3_raw)   / count(*), 1) AS pct_has_spinup,
           round(100.0 * count(smart_193_raw) / count(*), 1) AS pct_has_loadcycle,
           round(avg(capacity_bytes) / 1e12, 2) AS avg_tb
    FROM drivestats
    WHERE date = DATE '2026-03-15'
    GROUP BY 1
    HAVING count(smart_3_raw) = 0 OR count(smart_193_raw) = 0
    ORDER BY drives DESC
""").df().to_string(index=False))

## 5. Drive registry

One row per drive over the full window. This is cheap because it only touches a handful of
columns, and it gives us three things at once:

1. The model and capacity needed for **stratified sampling**.
2. `ever_failed`, the basis for cohort selection and labelling.
3. `last_seen`, which lets us distinguish a **failure** from **censoring**. Drives leave the
   fleet constantly through migration and decommissioning. A drive that simply stops
   appearing did *not* fail, and labelling its final days as negative would teach the model
   that pre-removal telemetry is healthy. `fleet_last_day` tells us whether a drive's
   disappearance is informative or just the edge of our window.

In [ ]:
t0 = time.time()
registry = con.sql(f"""
    SELECT
        serial_number,
        any_value(model)               AS model,
        max(capacity_bytes)            AS capacity_bytes,
        min(date)                      AS first_seen,
        max(date)                      AS last_seen,
        count(*)                       AS n_days,
        max(failure)                   AS ever_failed,
        max(CASE WHEN failure = 1 THEN date END) AS failure_date,
        any_value(datacenter)          AS datacenter,
        any_value(cluster_id)          AS cluster_id,
        any_value(vault_id)            AS vault_id
    FROM drivestats
    WHERE date BETWEEN DATE '{WARMUP_START}' AND DATE '{Q1_END}'
    GROUP BY serial_number
""").df()
print(f"registry built in {time.time() - t0:.1f}s -> {len(registry):,} drives")

registry.to_parquet(REGISTRY, index=False, compression="zstd")
print(f"saved {REGISTRY} ({os.path.getsize(REGISTRY) / 1e6:.1f} MB)")
registry.head()

In [ ]:
import pandas as pd

fleet_last_day = registry["last_seen"].max()

print(f"drives total      : {len(registry):,}")
print(f"failed drives     : {int(registry.ever_failed.sum()):,}")
print(f"fleet last day    : {fleet_last_day}")

# A drive vanishing before the window ends, without failure=1, was removed rather than failed.
left_early = (registry.last_seen < fleet_last_day) & (registry.ever_failed == 0)
print(f"censored (left early, no failure flag): {int(left_early.sum()):,}"
      f"  [{100 * left_early.mean():.1f}% of fleet]")

print("\n=== observation length distribution ===")
print(registry.n_days.describe().to_string())

print("\n=== top models ===")
print(registry.model.value_counts().head(12).to_string())

## 6. Cohort selection

**All** failed drives are kept, since failures are the scarce class (~1,030 in the quarter).
Healthy drives are sampled by **drive**, stratified by model so the rare-model tail is not
wiped out, and each carries a `sample_weight` equal to the inverse of its stratum's sampling
rate. Those weights are what let us report metrics at true fleet prevalence later despite
training on an enriched sample.

We also require a minimum observation length for healthy drives, so that every cohort member
can actually support a trailing window.

In [ ]:
import numpy as np

rng = np.random.default_rng(RANDOM_SEED)
MIN_DAYS_HEALTHY = 30

failed  = registry[registry.ever_failed == 1].copy()
healthy = registry[(registry.ever_failed == 0) & (registry.n_days >= MIN_DAYS_HEALTHY)].copy()

# Proportional allocation across model strata, with a floor so small models survive.
frac = min(1.0, N_HEALTHY_DRIVES / len(healthy))
picked = []
for model, grp in healthy.groupby("model", observed=True):
    take = max(min(len(grp), 25), int(round(len(grp) * frac)))
    take = min(take, len(grp))
    idx = rng.choice(grp.index.values, size=take, replace=False)
    sub = grp.loc[idx].copy()
    sub["sample_weight"] = len(grp) / take          # inverse sampling rate for this stratum
    picked.append(sub)

healthy_s = pd.concat(picked)
failed["sample_weight"] = 1.0                        # failures are fully enumerated

cohort = pd.concat([failed, healthy_s]).reset_index(drop=True)
cohort["is_failed"] = cohort.ever_failed.astype(int)

print(f"failed  drives : {len(failed):,}")
print(f"healthy drives : {len(healthy_s):,}  (sampled from {len(healthy):,})")
print(f"cohort total   : {len(cohort):,}")
print(f"expected panel rows ~ {cohort.n_days.sum():,}")

cohort.to_parquet(COHORT, index=False, compression="zstd")
print(f"\nsaved {COHORT}")

## 7. Column selection

We keep 24 SMART attributes with established diagnostic value rather than all 93, because
transfer volume is the binding constraint. Both the `_raw` and `_normalized` forms are kept
for the core failure indicators: the normalized value is what the drive compares against its
own failure threshold, so the pair carries information the raw value alone does not.

The four classic predictors are 5, 187, 197 and 198. Attributes 187/188 are **Seagate-reported
and absent on Toshiba and WDC** - roughly a two-thirds null rate fleet-wide. That is vendor-
conditional availability, not missing data, and notebook 03 handles it with explicit
missingness indicators. It must never be imputed.

In [ ]:
SMART_CORE = [5, 187, 188, 197, 198]              # reallocation / uncorrectable / pending
SMART_ERR  = [1, 7, 10, 183, 184, 199]            # read, seek, retry, bad block, e2e, CRC
SMART_WEAR = [4, 9, 12, 192, 193, 240, 241, 242]  # cycles, hours, shutdowns, LBAs
SMART_ENV  = [3, 189, 190, 191, 194]              # spin-up, high-fly, airflow, shock, temp

SMART_IDS = sorted(set(SMART_CORE + SMART_ERR + SMART_WEAR + SMART_ENV))
NORMALIZED_IDS = sorted(set(SMART_CORE + [1, 7, 10, 184, 199]))

META_COLS = ["date", "serial_number", "model", "capacity_bytes", "failure",
             "datacenter", "cluster_id", "vault_id", "pod_id", "pod_slot_num"]

smart_cols = [f"smart_{i}_raw" for i in SMART_IDS] + \
             [f"smart_{i}_normalized" for i in NORMALIZED_IDS]
PANEL_COLS = META_COLS + smart_cols

print(f"{len(SMART_IDS)} SMART attributes, {len(PANEL_COLS)} columns total "
      f"(vs 197 in the full schema)")
print("\nraw:", ", ".join(f"smart_{i}" for i in SMART_IDS))

## 8. Materialise the panel

The loop below writes **one parquet file per month** and skips months that already exist.
That makes it resumable: if Colab disconnects mid-extraction, re-running the cell picks up
where it stopped instead of starting over.

The cohort serials are registered as a local DuckDB table and joined server-side, so only
cohort rows cross the network. Expect roughly 5-20 minutes in total depending on bandwidth.

In [ ]:
con.register("cohort_serials", cohort[["serial_number", "sample_weight"]])
con.sql("CREATE OR REPLACE TABLE cohort_tbl AS SELECT * FROM cohort_serials")

MONTHS = [(2025, 12), (2026, 1), (2026, 2), (2026, 3)]
select_list = ",\n           ".join(f"d.{c}" for c in PANEL_COLS)

for year, month in MONTHS:
    out = f"{PANEL_DIR}/panel_{year}_{month:02d}.parquet"
    if os.path.exists(out):
        print(f"skip  {os.path.basename(out)}  (exists, {os.path.getsize(out)/1e6:.0f} MB)")
        continue

    start = date(year, month, 1)
    end   = date(year + (month == 12), (month % 12) + 1, 1)   # exclusive upper bound

    t0 = time.time()
    con.sql(f"""
        COPY (
            SELECT {select_list},
                   c.sample_weight
            FROM drivestats d
            JOIN cohort_tbl c USING (serial_number)
            WHERE d.date >= DATE '{start}' AND d.date < DATE '{end}'
        ) TO '{out}' (FORMAT parquet, COMPRESSION zstd);
    """)
    print(f"wrote {os.path.basename(out)}  "
          f"{os.path.getsize(out)/1e6:.0f} MB  in {time.time()-t0:.0f}s")

print("\npanel complete")

### 8b. Minimal full-fleet table

The counterfactual replay in Part 6 must run over the **real fleet**, not our enriched
sample, or the tome occupancy counts would be wrong and the safety invariant meaningless.
This table therefore covers every drive but carries only identity, topology and the failure
flag - no SMART columns - which keeps it small.

In [ ]:
for year, month in [(2026, 1), (2026, 2), (2026, 3)]:
    out = f"{FLEET_DIR}/fleet_{year}_{month:02d}.parquet"
    if os.path.exists(out):
        print(f"skip  {os.path.basename(out)}  (exists)")
        continue

    start = date(year, month, 1)
    end   = date(year + (month == 12), (month % 12) + 1, 1)

    t0 = time.time()
    con.sql(f"""
        COPY (
            SELECT date, serial_number, model, capacity_bytes, failure,
                   datacenter, cluster_id, vault_id, pod_id, pod_slot_num
            FROM drivestats
            WHERE date >= DATE '{start}' AND date < DATE '{end}'
        ) TO '{out}' (FORMAT parquet, COMPRESSION zstd);
    """)
    print(f"wrote {os.path.basename(out)}  "
          f"{os.path.getsize(out)/1e6:.0f} MB  in {time.time()-t0:.0f}s")

print("\nfleet table complete")

## 9. Validate the output

These assertions exist specifically to catch the failure mode that ruined the earlier
attempt at this dataset. The decisive metric is **observed days per drive**: if the panel is
a correct daily time series, healthy cohort drives should show something close to the full
window length. A median near 1-2 days would mean the time series had been destroyed by
row-level sampling, and every rolling feature built on top would be silently worthless.

In [ ]:
panel = pl.scan_parquet(f"{PANEL_DIR}/panel_*.parquet")

summary = panel.select(
    rows        = pl.len(),
    drives      = pl.col("serial_number").n_unique(),
    days        = pl.col("date").n_unique(),
    min_date    = pl.col("date").min(),
    max_date    = pl.col("date").max(),
    failures    = pl.col("failure").sum(),
).collect()
print(summary.to_pandas().T.rename(columns={0: "value"}).to_string())

per_drive = (panel.group_by("serial_number").agg(n = pl.len())
                  .collect()["n"].describe())
print("\n=== observed days per drive ===")
print(per_drive.to_pandas().to_string(index=False))

In [ ]:
stats = panel.group_by("serial_number").agg(n = pl.len()).collect()["n"]
median_days = stats.median()

print(f"median observed days per drive : {median_days}")
print(f"drives with only 1 day         : {(stats == 1).sum():,}  [{100*(stats==1).mean():.1f}%]")
print(f"drives with >= 90 days         : {(stats >= 90).sum():,}")

# The previous processed file had a median of 2 days and 47% single-day drives, which is what
# silently flattened every delta feature to zero. Anything in that regime is a hard failure.
assert median_days >= 60, (
    f"median {median_days} days/drive is far too low - the panel is row-sampled, not "
    "drive-sampled, and rolling features built on it would be meaningless"
)
print("\nPASS - per-drive daily time series is intact")

In [ ]:
total = 0
print(f"{'file':44s} {'MB':>9s}")
print("-" * 55)
for root, _, files in os.walk(DRIVE_ROOT):
    for f in sorted(files):
        if f.endswith(".parquet"):
            p = os.path.join(root, f)
            mb = os.path.getsize(p) / 1e6
            total += mb
            print(f"{os.path.relpath(p, DRIVE_ROOT):44s} {mb:9.1f}")
print("-" * 55)
print(f"{'TOTAL':44s} {total:9.1f} MB")

## Next steps

The panel is now a genuine per-drive daily time series on Drive, and the full-fleet topology
table is ready for the safety simulator.

- **`02_eda.ipynb`** - fleet characterisation, the vendor-conditional SMART availability
  matrix, failure rates by model / capacity / age, the bathtub curve, tome geometry, and
  pre-failure attribute trajectories.
- **`03_feature_engineering.ipynb`** - time-to-failure labels with censoring handled,
  strictly causal trailing-window features, tome-context features, the embargoed temporal
  split, and leakage tests.

One open question to settle during EDA: the quarter contains only ~1,030 failures, so the
test window will hold roughly 250. If bootstrap confidence intervals on PR-AUC come out too
wide to support a claim, extend training back through the 2025 quarters. That costs nothing
structurally - only `WARMUP_START` and the split dates in section 1 change, because the
Iceberg source is quarter-agnostic.